# 🏋️ Kaggle Training Pipeline — Business Entity Resolution

Runs the **full ML pipeline on Kaggle** using pre-cleaned datasets (no re-cleaning needed).

| Stage | Description |
|-------|-------------|
| 1 | Clone repo & install deps |
| 2 | Config + PipelineState init |
| 3 | Load pre-cleaned datasets |
| 4 | Train / Val split |
| 5 | 8-pass TF-IDF & Token Blocking |
| 6 | Blocking Recall evaluation |
| 7 | Hard-negative pair building |
| 8 | 30+ pairwise feature extraction |
| 9 | XGBoost training (GPU) |
| 10 | Validation scoring |
| 11 | Macro F0.5 threshold sweep |
| 12 | Error analysis |
| 13 | Test candidate generation |
| 14 | Test feature extraction |
| 15 | Test inference |
| 16 | Write submission TSVs |
| 17 | Inspect output files |


---
## 📦 Cell 1: Clone Repository & Install Dependencies

In [ ]:
import os

%cd /kaggle/working
!rm -rf ml_challenge
!git clone https://github.com/BharathPESU/ml_challenge.git
%cd /kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution
!git checkout Bharath.2004
!git pull origin Bharath.2004
%pip install -r requirements.txt --quiet

print("\n✓ Repository cloned and dependencies installed!")
print("✓ CWD:", os.getcwd())


---
## ⚙️ Cell 2: Initialise Config & PipelineState

In [ ]:
import sys, os, logging
from IPython.display import display

SRC = "/kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from config import Config
from pipeline import PipelineState

# ── Cleaned data (uploaded as Kaggle Dataset) ──────────────────────────────
CLEANED_TRAIN = "/kaggle/input/datasets/cdbharath234/cleaned_data/train"
CLEANED_TEST  = "/kaggle/input/datasets/cdbharath234/cleaned_data/test"

# Ground-truth lives in the original raw dataset
GT_PATH = (
    "/kaggle/input/datasets/cdbharath234/dataset3/student_resource/"
    "dataset/train/train_ground_truth.tsv"
)

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build config
cfg = Config()

# Cleaned training sources
cfg.data.train_source1_path       = os.path.join(CLEANED_TRAIN, "train_source1_clean.tsv")
cfg.data.train_source2_path       = os.path.join(CLEANED_TRAIN, "train_source2_clean.tsv")
cfg.data.train_source3_path       = os.path.join(CLEANED_TRAIN, "train_source3_clean.tsv")
cfg.data.train_ground_truth_path  = GT_PATH

# Cleaned test sources  (used later for submission inference)
cfg.data.test_source1_path        = os.path.join(CLEANED_TEST, "test_source1_clean.tsv")
cfg.data.test_source2_path        = os.path.join(CLEANED_TEST, "test_source2_clean.tsv")
cfg.data.test_source3_path        = os.path.join(CLEANED_TEST, "test_source3_clean.tsv")

cfg.data.output_dir               = OUTPUT_DIR
cfg.runtime.debug_sample_size     = None    # Full dataset run

# GPU settings for Kaggle P100/T4 — 'hist' works on both CPU and GPU
cfg.xgboost.tree_method = "hist"
cfg.xgboost.device      = "cuda"

cfg.configure_logging()
state = PipelineState()

print("✓ Config initialised")
print(f"  Cleaned train : {CLEANED_TRAIN}")
print(f"  Cleaned test  : {CLEANED_TEST}")
print(f"  Ground truth  : {GT_PATH}")
print(f"  Output dir    : {OUTPUT_DIR}")
print(f"  XGBoost       : device={cfg.xgboost.device}, tree_method={cfg.xgboost.tree_method}")


---
## 🔍 Cell 3: Verify All Input Files Exist

In [ ]:
files_to_check = [
    ("train_source1_clean", cfg.data.train_source1_path),
    ("train_source2_clean", cfg.data.train_source2_path),
    ("train_source3_clean", cfg.data.train_source3_path),
    ("train_ground_truth",  cfg.data.train_ground_truth_path),
    ("test_source1_clean",  cfg.data.test_source1_path),
    ("test_source2_clean",  cfg.data.test_source2_path),
    ("test_source3_clean",  cfg.data.test_source3_path),
]

all_ok = True
for name, path in files_to_check:
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1_048_576
        print(f"  ✅ {name:<28s}  {mb:7.1f} MB   {path}")
    else:
        print(f"  ❌ MISSING: {name:<22s}  {path}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "One or more input files are missing. "
        "Make sure the cleaned_data Kaggle dataset and the original dataset are both attached."
    )
print("\n✅ All input files found — proceeding!")


---
## 📥 Cell 4: Load Pre-Cleaned Training Datasets & Ground Truth

In [ ]:
from io_utils import read_tsv, validate_source_file, validate_ground_truth_file
from ground_truth import parse_ground_truth

print("⏳ Loading pre-cleaned training datasets...")
state.train_s1_clean = read_tsv(cfg.data.train_source1_path)
state.train_s2_clean = read_tsv(cfg.data.train_source2_path)
state.train_s3_clean = read_tsv(cfg.data.train_source3_path)

# Alias — pipeline stages that check state.train_sN need both
state.train_s1 = state.train_s1_clean
state.train_s2 = state.train_s2_clean
state.train_s3 = state.train_s3_clean

validate_source_file(state.train_s1_clean, "train_source1_clean")
validate_source_file(state.train_s2_clean, "train_source2_clean")
validate_source_file(state.train_s3_clean, "train_source3_clean")

gt_df = read_tsv(cfg.data.train_ground_truth_path)
validate_ground_truth_file(gt_df)
state.gt = parse_ground_truth(gt_df)

assert state.gt is not None
print("✅ Loaded pre-cleaned training data:")
print(f"   S1 clean : {len(state.train_s1_clean):>10,} rows")
print(f"   S2 clean : {len(state.train_s2_clean):>10,} rows")
print(f"   S3 clean : {len(state.train_s3_clean):>10,} rows")
print(f"   GT       : {len(state.gt):>10,} S1 entities")


---
## ✂️ Cell 5 — Stage 3: Train / Validation Split (80 / 20)

In [ ]:
from pipeline import split_training_validation

print("⏳ Stage 3: Splitting entities into train / val...")
split_training_validation(cfg, state)
assert state.train_s1_ids is not None and state.val_s1_ids is not None
print(f"✅ Stage 3 Complete — Train IDs: {len(state.train_s1_ids):,} | Val IDs: {len(state.val_s1_ids):,}")


---
## 🔍 Cell 6 — Stage 4: Generate Candidate Pairs (8-pass Blocking)

In [ ]:
from pipeline import generate_candidates

print("⏳ Stage 4: Running multi-pass TF-IDF + token blocking...")
generate_candidates(cfg, state)
assert state.train_candidates is not None and state.val_candidates is not None
print(f"✅ Stage 4 Complete — Train cands: {len(state.train_candidates):,} | Val cands: {len(state.val_candidates):,}")


---
## 📊 Cell 7 — Stage 5: Evaluate Blocking Recall

In [ ]:
from pipeline import evaluate_candidate_recall

print("⏳ Stage 5: Computing blocking recall...")
evaluate_candidate_recall(cfg, state)
print("✅ Stage 5 Complete — Recall numbers printed in logs above.")


---
## 🏗️ Cell 8 — Stage 6: Build Training Pairs & Hard Negatives

In [ ]:
from pipeline import build_training_pairs

print("⏳ Stage 6: Labelling candidates & sampling hard negatives...")
build_training_pairs(cfg, state)
assert state.train_pairs is not None and state.val_pairs is not None
print(f"✅ Stage 6 Complete — Train pairs: {len(state.train_pairs):,} | Val pairs: {len(state.val_pairs):,}")


---
## 🧮 Cell 9 — Stage 7: Extract 30+ Pairwise Similarity Features

In [ ]:
from pipeline import build_features

print("⏳ Stage 7: Computing pairwise features...")
build_features(cfg, state)
assert state.feature_columns is not None
print(f"✅ Stage 7 Complete — {len(state.feature_columns)} features: {state.feature_columns[:6]}...")


---
## 🤖 Cell 10 — Stage 8: Train XGBoost (GPU)

In [ ]:
from pipeline import train_xgboost

print("⏳ Stage 8: Training XGBoost classifier...")
try:
    train_xgboost(cfg, state)
except Exception as gpu_err:
    print(f"⚠️  GPU attempt failed ({gpu_err}). Falling back to CPU...")
    cfg.xgboost.device = "cpu"
    train_xgboost(cfg, state)

print(f"✅ Stage 8 Complete — Model saved to: {state.model_path}")


---
## 📈 Cell 11 — Stage 9: Score Validation Set

In [ ]:
from pipeline import score_validation

print("⏳ Stage 9: Scoring validation candidates...")
score_validation(cfg, state)
assert state.val_probabilities is not None
print(f"✅ Stage 9 Complete — Probabilities for {len(state.val_probabilities):,} val candidates")


---
## 🎯 Cell 12 — Stage 10: Macro F0.5 Threshold Sweep

In [ ]:
from pipeline import optimize_threshold

print("⏳ Stage 10: Sweeping thresholds 0.05 → 0.95 for best Macro F0.5...")
optimize_threshold(cfg, state)
assert state.best_threshold is not None
print(f"✅ Stage 10 Complete — Best threshold: {state.best_threshold:.4f}")


---
## 🔎 Cell 13 — Stage 11: Error Analysis

In [ ]:
from pipeline import perform_error_analysis

print("⏳ Stage 11: Analysing FP / FN errors...")
perform_error_analysis(cfg, state)
print("✅ Stage 11 Complete — See logs above for FP / FN breakdown.")


---
## 🧪 Cell 14 — Stage 12: Generate Test Candidates (Pre-Cleaned Test Data)

In [ ]:
from inference import generate_test_candidates as _gen_test_cands
from io_utils import read_tsv

print("⏳ Stage 12: Loading pre-cleaned test data...")
state.test_s1_clean = read_tsv(cfg.data.test_source1_path)
state.test_s2_clean = read_tsv(cfg.data.test_source2_path)
state.test_s3_clean = read_tsv(cfg.data.test_source3_path)
print(f"   test_S1: {len(state.test_s1_clean):,} | test_S2: {len(state.test_s2_clean):,} | test_S3: {len(state.test_s3_clean):,}")

print("⏳ Stage 12: Generating test candidate pairs (blocking)...")
bc = cfg.blocking
blocking_config = {
    "name_top_k":             bc.name_top_k,
    "address_top_k":          bc.address_top_k,
    "min_name_similarity":    bc.min_name_similarity,
    "min_address_similarity": bc.min_address_similarity,
    "tfidf_analyzer":         bc.tfidf_analyzer,
    "tfidf_ngram_range":      list(bc.tfidf_ngram_range),
    "tfidf_min_df":           bc.tfidf_min_df,
    "tfidf_sublinear_tf":     bc.tfidf_sublinear_tf,
    "use_token_blocking":     True,
}
state.test_candidates = _gen_test_cands(
    state.test_s1_clean, state.test_s2_clean, state.test_s3_clean,
    blocking_config=blocking_config,
)
assert state.test_candidates is not None
print(f"✅ Stage 12 Complete — Test candidates: {len(state.test_candidates):,}")


---
## 🧮 Cell 15 — Stage 13: Build Test Features

In [ ]:
from pipeline import build_test_features

print("⏳ Stage 13: Computing test feature similarities...")
build_test_features(cfg, state)
assert state.test_features is not None
print(f"✅ Stage 13 Complete — Test feature matrix: {state.test_features.shape}")


---
## 🎯 Cell 16 — Stage 14: Run Test Inference

In [ ]:
from pipeline import run_test_inference

print("⏳ Stage 14: Scoring test candidates with trained model...")
run_test_inference(cfg, state)
assert state.test_predictions is not None
n_matched = sum(1 for v in state.test_predictions.values() if v)
print(f"✅ Stage 14 Complete — {len(state.test_predictions):,} S1 entities | {n_matched:,} with matches")


---
## 📝 Cell 17 — Stage 15: Generate Submission TSV Files

In [ ]:
from pipeline import create_submission_files

print("⏳ Stage 15: Writing submission files...")
create_submission_files(cfg, state)
print(f"\n🎉 Stage 15 Complete — Submission files in: {cfg.data.output_dir}")
print(f"   📄 {cfg.data.output_dir}/matching_results.tsv")
print(f"   📄 {cfg.data.output_dir}/candidate_pairs.tsv")


---
## ✅ Cell 18: Inspect & Validate Submission Files

In [ ]:
import pandas as pd
from IPython.display import display

matching_path   = os.path.join(cfg.data.output_dir, "matching_results.tsv")
candidates_path = os.path.join(cfg.data.output_dir, "candidate_pairs.tsv")

if os.path.exists(matching_path) and os.path.exists(candidates_path):
    m_df = pd.read_csv(matching_path,   sep="\t", dtype=str, keep_default_na=False)
    c_df = pd.read_csv(candidates_path, sep="\t", dtype=str, keep_default_na=False)

    print("📊 matching_results.tsv")
    print(f"   Total S1 entities : {len(m_df):,}")
    print(f"   With matches      : {(m_df['matched_entity_ids'] != '').sum():,}")
    print(f"   Singletons        : {(m_df['matched_entity_ids'] == '').sum():,}")
    print()
    print("📊 candidate_pairs.tsv")
    print(f"   Total pairs       : {len(c_df):,}")
    print()
    print("First 10 rows of matching_results.tsv:")
    display(m_df.head(10))
else:
    print("⚠️  Submission files not found — ensure Cell 17 completed successfully.")
